# Voice Emotion Training

Train the audio emotion classifier saved to models/voice_emotion.pkl used by the voice API endpoint.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


## Data requirements

Expected WAV layout: data/raw/voice/{happy,sad,angry,neutral,fearful}/*.wav. Use scripts/prepare_crema_d_av.py to build this from CREMA-D if needed.


## Dataset inventory


In [ ]:
from pathlib import Path

voice_root = repo_root / 'data' / 'raw' / 'voice'
print('Voice root:', voice_root, 'exists=', voice_root.exists())
if voice_root.exists():
    for cls in ['happy', 'sad', 'angry', 'neutral', 'fearful']:
        cnt = sum(1 for p in (voice_root / cls).rglob('*.wav')) if (voice_root / cls).exists() else 0
        print(cls, 'wav files:', cnt)


## Links to code


- Training script: `app/models/voice/emotion_train.py`


- Feature extraction: `app/models/voice/features.py`


- Inference module: `app/models/voice/emotion_predict.py`


- API endpoint: `POST /api/voice/emotion`


In [ ]:
# Update paths if your data lives elsewhere.



In [ ]:
voice_dir = repo_root / 'data' / 'raw' / 'voice'
print('Voice dir exists:', voice_dir.exists(), voice_dir)


## Optional: prepare CREMA-D dataset


In [ ]:
# !python scripts/prepare_crema_d_av.py --limit 500


## Train the model


In [ ]:
# !python app/models/voice/emotion_train.py --limit-per-class 500 --min-per-class 25


## Quick inference check


In [ ]:
from app.models.voice.emotion_predict import predict_emotion

sample = None
if voice_dir.exists():
    for wav in voice_dir.rglob('*.wav'):
        sample = wav
        break

if sample is None:
    print('No sample wav found under data/raw/voice')
else:
    out = predict_emotion(audio_bytes=sample.read_bytes(), filename=sample.name)
    print(out)


## Visuals and metrics


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from app.models.voice.features import load_audio
from app.models.voice.emotion_predict import predict_emotion

voice_root = repo_root / 'data' / 'raw' / 'voice'
sample = None
if voice_root.exists():
    for p in voice_root.rglob('*.wav'):
        sample = p
        break

if sample is None:
    print('No WAV sample found under', voice_root)
else:
    audio, sr = load_audio(sample)
    pred = predict_emotion(audio_bytes=sample.read_bytes(), filename=sample.name)
    print('Prediction:', pred)
    plt.figure(figsize=(8, 2))
    plt.plot(audio)
    plt.title('Waveform: ' + sample.name)
    plt.tight_layout()
    plt.show()
    try:
        import librosa
        import librosa.display
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        plt.figure(figsize=(6, 3))
        librosa.display.specshow(mfcc, x_axis='time')
        plt.title('MFCC')
        plt.colorbar()
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('MFCC plot skipped (librosa unavailable):', exc)
